#### 2.1. Qwen3 and the Reasoning From Scratch package

In [1]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [2]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

#### 2.2. Encoding and Decoding

In [3]:
prompt = "Explain the large language models."
input_token_ids_list = tokenizer.encode(prompt)
input_token_ids_list

[840, 20772, 279, 3460, 4128, 4119, 13]

In [4]:
for i in input_token_ids_list:
    print(f"{i:5d} --> `{tokenizer.decode([i])}`")

  840 --> `Ex`
20772 --> `plain`
  279 --> ` the`
 3460 --> ` large`
 4128 --> ` language`
 4119 --> ` models`
   13 --> `.`


#### 2.3. Downloading weights and selecting a device

In [5]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3\qwen3-0.6B-base.pth already up-to-date


In [6]:
import torch

def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("using NVIDIA CUDA GPU")
    
        if enable_tensor_cores: 
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 11):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True
    
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("using Apple MPS GPU")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("using Intel XPU GPU")
    
    else:
        device = torch.device("cpu")
        print("using CPU")
    
    return device

device = get_device()

using NVIDIA CUDA GPU


In [7]:
# device = torch.device("cpu")
# print("using CPU")

#### 2.4. Loading the pretrained Qwen3 model

In [8]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device);

C:\Users\Eliey\AppData\Local\Temp\ipykernel_6600\2252560431.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


#### 2.5. Input tensors and batch dimensions

In [9]:
prompt = "Explain large language models."
input_tokens_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_tokens_ids_list)}")

input_tensor = torch.tensor(input_tokens_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


#### 2.6. Building a streaming text generator

In [10]:
@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        # use the last token as answer
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # stop if encounter an end-of-sequence token
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        # yield each token as it's generated
        yield next_token 

        token_ids = torch.cat([token_ids, next_token], dim=-1)

In [11]:
prompt = "Explain large lanugage models."

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
    ).unsqueeze(0)
max_new_tokens = 5

for token in generate_text_basic_stream(model, input_token_ids_tensor, max_new_tokens, eos_token_id=tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist()
    
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
        )

 What are the advantages and

#### 2.7. Benchmarking text generation

In [12]:
import warnings

def generate_stats(output_token_ids, start_time, end_time):
    total_time = end_time - start_time

    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)), ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():
            # check whether we actually use this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} backend is available but not used. The tensor is on {device_type}."
                )
            
            # synchchronize the backend to get accurate timing
            if hasattr(backend, "synchronize"):
                backend.synchronize()

            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024**3)
            print(f"{name} Max Memory Allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [13]:
import time 

start_time = time.time()
generate_ids = []

for token in generate_text_basic_stream(
    model, input_token_ids_tensor, max_new_tokens, eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    next_token_id = token.squeeze(0)
    generate_ids.append(next_token_id)

end_time = time.time()

output_token_ids_tensor = torch.cat(generate_ids, dim=0)
generate_stats(output_token_ids_tensor, start_time, end_time)

 What are the advantages and

Time: 0.33 sec
15 tokens/sec
CUDA Max Memory Allocated: 1.46 GB


#### 2.8. Adding KV caching and measuring the speedup 

In [14]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1]
    for step in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            out = model(next_token, cache=cache)[:, -1]

In [15]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, start_time, end_time)

 What are the advantages and

Time: 0.39 sec
12 tokens/sec
CUDA Max Memory Allocated: 1.45 GB


#### 2.9. Model compilation with torch.compile

In [16]:
from pathlib import Path
import torch

from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)

model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)

model.to(device)
model.eval()

print("Model type:", type(model))
print("Model loaded successfully!")

Model type: <class 'reasoning_from_scratch.qwen3.Qwen3Model'>
Model loaded successfully!


In [17]:
model.eval()

compiled_model = torch.compile(model)

In [18]:
for i in range(3):
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream(
        model=compiled_model,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(tokenizer.decode(token_id), end="", flush=True)
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, start_time, end_time)

    print(f"\n{30*'-'}\n")

BackendCompilerFailed: backend='inductor' raised:
RuntimeError: Cannot find a working triton installation. More information on installing Triton can be found at https://github.com/openai/triton

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
